In [ ]:

!pip install -q -U transformers datasets


['negative' 'neutral' 'positive']
3876 970
['negative' 'neutral' 'positive']


In [ ]:
#load raw data
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/riccardotella/Sentiment_Analysis_of_Financial_News/main/data/processed/preprocessed_texts.csv")

X = df["text"]
y = df["label"]

#encoding labels to integers
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_enc = le.fit_transform(y)      # negative/neutral/positive -> 0/1/2
print(le.classes_)                # remember which int = which class

#split 80/20
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

print(len(X_train), len(X_test))
print(le.classes_)

In [ ]:
# tokenize ( DistilBERT)
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# example
example = X_train.iloc[0]
print("ORIGINAL:", example)
print("TOKENS:", tokenizer.tokenize(example))
print("IDS:", tokenizer(example)["input_ids"])



ModuleNotFoundError: No module named 'transformers'

In [ ]:
#tokenize datasets
from datasets import Dataset

# build a Dataset from your text + labels (do this for train and test)
train_ds = Dataset.from_dict({"text": list(X_train), "labels": list(y_train)})
test_ds  = Dataset.from_dict({"text": list(X_test),  "labels": list(y_test)})

# a function that tokenizes a batch of examples
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

# apply it to both datasets
train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)


In [ ]:
#load prtrained model
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)


In [ ]:
# TRAINING setting up

# 1.Metrics function

import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)        # logits -> predicted class
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

# 2. Training Arguments and TRainer

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_steps=50,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)



In [ ]:
# training

trainer.train()

In [ ]:
# evaluation
results = trainer.evaluate()
print(results)


In [ ]:
#perclass report and confusion matrix

import numpy as np
preds_output = trainer.predict(test_ds)
y_pred_t5 = np.argmax(preds_output.predictions, axis=1)

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred_t5, target_names=le.classes_))
print(confusion_matrix(y_test, y_pred_t5))

